# Runnable Streaming Event Schemas

The `schema.py` module defines the typed dictionary structures used by the `Runnable.astream_events` interface.

It separates the data carried by an event from the common event metadata and distinguishes standard LangChain lifecycle events from user-created custom events.

# EventData

`EventData` defines the payload associated with a standard Runnable streaming event.

The dictionary uses `total=False`, so every field is optional. The fields available for a particular event depend on whether the event represents the start, streamed output, successful completion, or failure of a Runnable.

## Bases

- `TypedDict`

## Attributes

1. `input`: Stores the input passed to the Runnable.

   The input may be available at the start or end of execution. When the Runnable streams its input, the complete input is generally unavailable until input streaming has finished.

   * **Type:**
     ```python
     input: Any
     ```

2. `error`: Stores the exception raised during Runnable execution.

   This field is available only for events representing failed execution.

   * **Type:**
     ```python
     error: NotRequired[
         BaseException
     ]
     ```

3. `output`: Stores the completed output produced by the Runnable.

   This field is available at the end of successful execution. For most Runnables, the completed output can be reconstructed by combining the values emitted through `chunk`.

   * **Type:**
     ```python
     output: Any
     ```

4. `chunk`: Stores one streaming output chunk.

   Streaming chunks generally support addition, allowing them to be combined into the completed Runnable output.

   * **Type:**
     ```python
     chunk: Any
     ```

5. `tool_call_id`: Stores the identifier of the tool call associated with a tool execution.

   This field is available for `on_tool_error` events and can associate an error with a particular tool call.

   * **Type:**
     ```python
     tool_call_id: NotRequired[
         str | None
     ]
     ```

# BaseStreamEvent

`BaseStreamEvent` defines the fields shared by standard and custom streaming events.

Standard lifecycle event names generally follow the format `on_<runnable_type>_<stage>`, where the stage is `start`, `stream`, or `end`.

## Bases

- `TypedDict`

## Attributes

1. `event`: Stores the event name.

   Standard events identify both the Runnable category and lifecycle stage.

   * **Type:**
     ```python
     event: str
     ```

2. `run_id`: Stores the unique identifier assigned to the Runnable execution.

   Every child Runnable execution receives its own identifier.

   * **Type:**
     ```python
     run_id: str
     ```

3. `tags`: Stores optional tags associated with the Runnable execution.

   Tags inherited from parent Runnables may also appear in this list.

   * **Type:**
     ```python
     tags: NotRequired[
         list[str]
     ]
     ```

4. `metadata`: Stores optional metadata associated with the Runnable execution.

   Metadata may be bound to a Runnable or supplied through runtime configuration.

   * **Type:**
     ```python
     metadata: NotRequired[
         dict[str, Any]
     ]
     ```

5. `parent_ids`: Stores the execution identifiers of the event's parent Runnables.

   Identifiers are ordered from the root parent to the immediate parent. Root events use an empty sequence. Parent identifiers are populated by version 2 of the streaming-events API.

   * **Type:**
     ```python
     parent_ids: Sequence[str]
     ```

# StandardStreamEvent

`StandardStreamEvent` represents a LangChain-defined Runnable lifecycle event.

It extends the common event fields with a structured `EventData` payload and the name of the Runnable that generated the event.

## Bases

- `BaseStreamEvent`

## Attributes

1. `data`: Stores the event-specific payload.

   Its populated fields depend on the event type and lifecycle stage.

   * **Type:**
     ```python
     data: EventData
     ```

2. `name`: Stores the name of the Runnable that generated the event.
   * **Type:**
     ```python
     name: str
     ```

# CustomStreamEvent

`CustomStreamEvent` represents a user-defined streaming event.

Its event name is always `on_custom_event`, while its `name` and `data` values are supplied by the user.

## Bases

- `BaseStreamEvent`

## Attributes

1. `event`: Stores the fixed custom-event identifier.
   * **Type:**
     ```python
     event: Literal[
         "on_custom_event"
     ]
     ```

2. `name`: Stores the user-defined name of the custom event.
   * **Type:**
     ```python
     name: str
     ```

3. `data`: Stores the user-defined custom-event payload.

   The payload is unrestricted and may contain any value.

   * **Type:**
     ```python
     data: Any
     ```

## Type Aliases

1. `StreamEvent`: Represents either a standard LangChain streaming event or a user-defined custom event.
   * **Definition:**
     ```python
     StreamEvent = StandardStreamEvent | CustomStreamEvent
     ```